# Interactive Notebook 02 - PWM extensions:



### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})
import jax
import jax.numpy as jnp

In [ ]:
from analytical_harmonics import build_analytical_spectrum, visualize_analytical_spectrum
from helper_functions import triangular_signal, compute_three_phase_signals, three_phase_plot, plot_fft_spectrum, get_fft_spectrum

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

### Three-phase harmonics:

In [ ]:
T_full = 0.2 # s
N = 10_000
f_s = N / T_full  # 'N' samples per 'T_full' seconds

carrier_freq = 1000  # Hz
T_p = 1 / carrier_freq
t = jnp.linspace(0, T_full, N)
print(t.shape)

In [ ]:
c_t = triangular_signal(t, frequency=carrier_freq, amplitude=1.0, phase=0.0)

u_dc = 20  # V
m = 1.0
u_ref_freq = 10  # Hz
u_ref_t, s_ref_t, s_t = compute_three_phase_signals(m=m, u_ref_freq=u_ref_freq, t=t, c_t=c_t, u_dc=u_dc)
fig, axs = three_phase_plot(t, s_ref_t, c_t, s_t)

#### FFT-based harmonics:

In [ ]:
# test FFT settings on single sine wave:
spectrum = jnp.abs(jnp.fft.rfft(s_ref_t[..., 0], axis=0))
freqs = jnp.fft.rfftfreq(N, d=1/f_s)
amps  = (2 / N) * jnp.abs(spectrum)

plt.bar(freqs[:150] / u_ref_freq, amps[:150])
plt.xlim(0, 60)

In [ ]:
# single-phase PWM harmonics:
plot_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)

In [ ]:
# exemplary line-to-line PWM harmonics:
plot_fft_spectrum(s_t[..., 0] - s_t[..., 1], f_s, N, u_ref_freq)

#### Comparison of analytical and numeric spectra:

In [ ]:
fig, axs = plot_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)

## 

spectrum = build_analytical_spectrum(
    pulse_number=carrier_freq/u_ref_freq,  # 100
    modulation_index=m,
    fundamental_frequency_hz=u_ref_freq,  # 10
)

frequencies = []
amplitudes = []
for freq_dict in spectrum:
    frequencies.append(freq_dict["normalized_frequency"])
    amplitudes.append(freq_dict["amplitude"])

for ax in axs:
    ax.bar(frequencies, amplitudes, alpha=0.5)

The FFT-based spectrum fits to the spectrum based on analytical computations. We will consider only FFT-based spectra in the following.

#### Total harmonic distortion:

The total harmonic distortion (THD) measures the ratio of the harmonic contribution (=? power) of the fundamental frequency compared to all other harmonic content:

$\mathrm{THD} = \sqrt{\sum^{\infty}_{n=3,5,7,\dots} U_n^2 / U_1^2}$,

where $U_1$ is the amplitude of the fundamental frequency and $U_n$ is the amplitude of the $n$-th harmonic.
Furthermore, the assumption is made that 

In [ ]:
def get_amplitude_at_freq(amps, freqs, target_freq):
    idx = jnp.argmin(freqs - target_freq)
    return amps[idx]

def compute_THD(amps, freqs, f_fundamental):
    amp_fundamental = get_amplitude_at_freq(amps, freqs, f_fundamental)


    raise NotImplementedError("redo as matrix operation")
    for n in jnp.arange(3, 300, 2):
        amp_harmonic = get_amplitude_at_freq(amps, freqs, f_fundamental)
    

In [ ]:
amps, freqs = get_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)

In [ ]:
compute_THD(amps, freqs, u_ref_freq)

### Zero sequence injection:

Allow the extension of the modulation index $m$ from $1$ to $2 / \sqrt{3} \approx 1.155$

with min-max zero-sequence injection:

$u_0 (t) = \frac{1}{2} (\mathrm{max} \{u^*_{\mathrm{a}}, u^*_{\mathrm{b}}, u^*_{\mathrm{c}}\} + \mathrm{min} \{u^*_{\mathrm{a}}, u^*_{\mathrm{b}}, u^*_{\mathrm{c}}\})$

In [ ]:
def compute_three_phase_signals_zsi_minmax(m, u_ref_freq, t, c_t, u_dc):

    omega = u_ref_freq * 2 * jnp.pi
    s_ref_t = jnp.array(
        [
            m * jnp.sin(omega * t),
            m * jnp.sin(omega * t - jnp.pi * 2 / 3),
            m * jnp.sin(omega * t + jnp.pi * 2 / 3),
        ]
    ).T
    
    s_0_t = 0.5 * (jnp.max(s_ref_t, axis=-1) + jnp.min(s_ref_t, axis=-1))
    s_ref_t = s_ref_t - s_0_t[..., None]
    u_ref_t = u_dc / 2  * s_ref_t
    
    s_t = jnp.where(s_ref_t > c_t[..., None], 1, -1)

    return u_ref_t, s_ref_t, s_t, s_0_t

In [ ]:
u_ref_t, s_ref_t, s_t, s_0_t = compute_three_phase_signals_zsi_minmax(
    m=2/jnp.sqrt(3),
    u_ref_freq=u_ref_freq,
    t=t,
    c_t=c_t,
    u_dc=u_dc
)
fig, axs = three_phase_plot(t, s_ref_t, c_t, s_t, s_0_t)
plt.show()
plot_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)
plt.show()

### Clipping for overmodulation operation:

In [ ]:
def compute_three_phase_signals_zsi_clipping(m, u_ref_freq, t, c_t, u_dc):

    omega = u_ref_freq * 2 * jnp.pi
    s_ref_t = jnp.array(
        [
            m * jnp.sin(omega * t),
            m * jnp.sin(omega * t - jnp.pi * 2 / 3),
            m * jnp.sin(omega * t + jnp.pi * 2 / 3),
        ]
    ).T
    
    s_0_t = 1/2 * (jnp.max(s_ref_t, axis=-1) + jnp.min(s_ref_t, axis=-1))
    s_ref_t = jnp.clip(s_ref_t - s_0_t[..., None], min=-1, max=1)
    u_ref_t = u_dc / 2  * s_ref_t
    
    s_t = jnp.where(s_ref_t > c_t[..., None], 1, -1)

    return u_ref_t, s_ref_t, s_t, s_0_t

In [ ]:
u_ref_t, s_ref_t, s_t, s_0_t = compute_three_phase_signals_zsi_clipping(
    m=1.24,
    u_ref_freq=u_ref_freq,
    t=t,
    c_t=c_t,
    u_dc=u_dc
)
fig, axs = three_phase_plot(t, s_ref_t, c_t, s_t, s_0_t)
plt.show()
plot_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)
plt.show()

At maximum modulation index: $m = 4/3$.

In [ ]:
u_ref_t, s_ref_t, s_t, s_0_t = compute_three_phase_signals_zsi_clipping(
    m=4/3,
    u_ref_freq=u_ref_freq,
    t=t,
    c_t=c_t,
    u_dc=u_dc
)
fig, axs = three_phase_plot(t, s_ref_t, c_t, s_t, s_0_t)
plt.show()
plot_fft_spectrum(s_t[..., 0], f_s, N, u_ref_freq)
plt.show()

### 6-step operation:

6-step operation is utilized at the voltage and current limit (generally for maximum speed operation).
Note that for this operation, no PWM is necessary: The switches are operated open-loop solely based on the momentary electrical angle of the machine.

In practice, the electrical angular velcoity is based on the closed loop process, where the motor interacts with the mechanical load to produce the rotation.
Here, we assume a constant electrical angular frequency for simplicity.

In [ ]:
f_el = 25  # Hz
omega_el = 2 * jnp.pi * f_el

In [ ]:
eps_el = (omega_el * t) % (2 * jnp.pi)

sector = (eps_el / (jnp.pi / 3)).astype(int) % 6

switching_table = jnp.array([
    [ 1, -1, -1],
    [ 1,  1, -1],
    [-1,  1, -1],
    [-1,  1,  1],
    [-1, -1,  1],
    [ 1, -1,  1],
])

s_t = switching_table[sector]

In [ ]:
fig_left, axs = plt.subplots(
    6, 1, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [3, 2, 1, 1, 1, 1.5]}, constrained_layout=True
)
axs[0].plot(t, eps_el, color="tab:blue")


axs[1].step(t, sector, color="tab:blue")
axs[2].step(t, s_t[..., 0], color="tab:blue")
axs[3].step(t, s_t[..., 1], color="tab:red")
axs[4].step(t, s_t[..., 2], color="black")
axs[5].step(t, (s_t[..., 0] - s_t[..., 1]) / 2, color="tab:orange")


for ax in axs:
    ax.grid(alpha=0.5)
    ax.tick_params(which="major", axis="y", direction="in")
    ax.tick_params(which="both", axis="x", direction="in")
    ax.set_xlim(t[0], t[-1])


for ax in axs[2:]:
    ax.set_ylim(-1.1, 1.1)

axs[0].set_ylabel("$\\varepsilon_{\\mathrm{el}}$")
axs[1].set_ylabel("sector")
axs[2].set_ylabel("$s_\\mathrm{a}(t)$")
axs[3].set_ylabel("$s_\\mathrm{b}(t)$")
axs[4].set_ylabel("$s_\\mathrm{c}(t)$")
axs[5].set_ylabel("$\\frac{u_\\mathrm{a-b}(t)}{u_\\mathrm{dc}}$")
axs[-1].set_xlabel("$t$ in $s$")

plt.show()

### Interactive plot PWM:

Measure the TDD or THD and the output voltages / power, effective voltage value?